<a href="https://colab.research.google.com/github/AhmedWaleedElFar/flyrank-ml-assignments/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AhmedWaleedElFar/flyrank-ml-assignments/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

- One row = one pseudomnyzed content page with features aggregated from prior 90-day window (t-90 to t) and a target label predicting the decline in the next 30 days (t to t+30)

- Tables Included: fact_content_daily_performance (features), dim_content (metadata), dim_clients (history coverage).

- Tables Excluded: fact_content_query_90d (fixed 90-day window overlaps label window, which will lead to leakage).

- Decision date at t=2026-03-01, Features dates cover: 2025-12-01 to 2026-03-01 (t-90 to t), Reference window: 2026-02-01 to 2026-02-28 (t-30 to t), and target label covers: 2026-03-01 to 2026-03-31 (t to t+30).

- Target Label: `decline_in_next_30d = 1 if sum(impressions)_t_to_t+30 < 0.8 * sum(impressions)_t-30_to_t else 0`.


- Decision date t=2026-03-01. Features window: 2025-12-01 to 2026-03-01 (t-90 to t). Reference window: 2026-02-01 to 2026-02-28 (t-30 to t). Label window: 2026-03-01 to 2026-03-31 (t to t+30).

- Target Label: `decline_in_next_30d = 1 if sum(impressions)_t_to_t+30 < 0.8 * sum(impressions)_t-30_to_t else 0`.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

| Category | Fields |
|----------|--------|
| **Features** (5) | impressions_90d, clicks_90d, avg_position_90d, days_since_last_update, content_type |
| **Label** | decline_in_next_30d |
| **Context** | content_hash_id, client_hash_id, report_date, gsc_data_available, ga4_data_available |
| **Excluded** | fact_query_90d columns (leakage), ctr_90d (noisy), content_age_days (redundant with staleness), has_ai_traffic (sparse), word_count/char_count (missingness), search_volume/competition/cpc (missing for feedly content) |

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Verification checks (month=2026-03)

In [1]:
%pip -q install duckdb huggingface_hub

In [4]:
import os, getpass

# Token order: env var -> Colab Secret -> prompt (last resort).
# Use a Colab Secret named HF_TOKEN (the key panel on the left) so the prompt never
# fires: if Colab reconnects while a getpass prompt is open, the kernel waits on it
# forever ('Resuming execution...') and you have to restart the runtime.
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')


In [5]:
import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':                f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':                f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':                 f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample':          f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':             f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

for name, src in TABLES.items():
    n = con.sql(f'SELECT COUNT(*) FROM {src}').fetchone()[0]
    print(f'{name:22} {n:>12,} rows')

dim_clients                     104 rows
dim_content                 519,606 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_daily               78,835,655 rows
fact_daily_sample        11,694,072 rows
fact_query_90d            2,414,248 rows


Query 1: Grain Check

The table grain is report_date × client_hash_id × content_hash_id (one row per page per day per client).

In [6]:
query1 = con.sql(f"""
SELECT report_date, client_hash_id, content_hash_id, COUNT(*) c
FROM {TABLES["fact_daily"]}
WHERE month = '2026-03'
GROUP BY 1, 2, 3
HAVING c > 1
LIMIT 5;  -- Expect 0 rows (grain holds
""").df()

query1

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,c


Query 2: Counts + Date Span

In [7]:
query2 = con.sql(f"""
      SELECT
          COUNT(*) AS rows,
          MIN(report_date) AS min_date,
          MAX(report_date) AS max_date,
          COUNT(DISTINCT content_hash_id) AS unique_pages,
          COUNT(DISTINCT client_hash_id) AS unique_clients
      FROM {TABLES["fact_daily"]}
      WHERE month = '2026-03';
""").df()

query2

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,rows,min_date,max_date,unique_pages,unique_clients
0,9841378,2026-03-01,2026-03-31,331437,55


Query 3: Availability (IS TRUE)

In [8]:
query3 = con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END) AS gsc_true,
    SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS ga4_true,
    SUM(CASE WHEN gsc_data_available IS TRUE AND ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS both_true
FROM {TABLES['fact_daily']}
WHERE month = '2026-03';
""").df()

query3

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,gsc_true,ga4_true,both_true
0,9841378,3611061.0,413966.0,364347.0


Feature Frame Query (t=2026-03-01)

In [13]:
feature_frame = con.sql(f"""
WITH bounds AS (SELECT DATE '2026-03-01' AS t),
feat AS (
    SELECT content_hash_id,
           SUM(gsc_impressions) AS impressions_90d,
           SUM(gsc_clicks) AS clicks_90d,
           AVG(gsc_avg_position) AS avg_position_90d,
           MAX(CASE WHEN sessions_ai > 0 THEN 1 ELSE 0 END) AS has_ai_traffic  -- for completeness, not in final 5
    FROM {TABLES['fact_daily']}, bounds
    WHERE report_date >= t - INTERVAL 90 DAY
      AND report_date < t
      AND gsc_data_available IS TRUE
    GROUP BY 1
),
meta AS (
    SELECT content_hash_id,
           DATE_DIFF('day', content_created_date, DATE '2026-03-01') AS content_age_days,
           DATE_DIFF('day', content_updated_date, DATE '2026-03-01') AS days_since_last_update,
           content_type
    FROM {TABLES['dim_content']}
),
labels AS (
    SELECT content_hash_id,
           SUM(gsc_impressions) AS imp_next30
    FROM {TABLES['fact_daily']}, bounds
    WHERE report_date >= t
      AND report_date < t + INTERVAL 30 DAY
      AND gsc_data_available IS TRUE
    GROUP BY 1
),
ref AS (
    SELECT content_hash_id,
           SUM(gsc_impressions) AS imp_prior30
    FROM {TABLES['fact_daily']}, bounds
    WHERE report_date >= t - INTERVAL 30 DAY
      AND report_date < t
      AND gsc_data_available IS TRUE
    GROUP BY 1
)
SELECT f.*, m.content_age_days, m.days_since_last_update, m.content_type,
       r.imp_prior30, l.imp_next30,
       CASE WHEN l.imp_next30 < 0.8 * r.imp_prior30 THEN 1 ELSE 0 END AS decline_in_next_30d
FROM feat f
JOIN meta m USING (content_hash_id)
JOIN ref r USING (content_hash_id)
JOIN labels l USING (content_hash_id);
""").df()

feature_frame

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,content_hash_id,impressions_90d,clicks_90d,avg_position_90d,has_ai_traffic,content_age_days,days_since_last_update,content_type,imp_prior30,imp_next30,decline_in_next_30d
0,content_14a3d47ccd0d15dc,8.0,0.0,6.285714,0,82,-72,keyword article,5.0,2.0,1
1,content_14a6f92117604fef,50.0,0.0,9.358333,0,80,-80,keyword article,34.0,88.0,0
2,content_14a86c63a214f648,72.0,0.0,19.160794,0,82,-72,keyword article,14.0,42.0,0
3,content_14b1a02c1b8557fb,20.0,1.0,16.000000,0,80,-72,keyword article,6.0,90.0,0
4,content_14c17f59aa610ab3,152.0,2.0,5.588385,0,12,-113,keyword article,152.0,113.0,1
...,...,...,...,...,...,...,...,...,...,...,...
134394,content_170d99774138d1cb,9.0,2.0,4.857143,0,208,-80,feedly article,2.0,3.0,0
134395,content_17145e4136a3dea1,15.0,0.0,4.555556,0,182,-80,feedly article,11.0,8.0,1
134396,content_171516ac5054f8c1,21.0,1.0,4.431373,0,198,-80,feedly article,7.0,1.0,1
134397,content_1715ff8eeafc7050,13.0,0.0,5.333333,0,199,-80,feedly article,5.0,8.0,0


Features selected:


| Feature | Knowable at decision moment because... |
|---------|----------------------------------------|
| `impressions_90d` | Aggregated from t-90 to t (Dec 2025–Feb 2026), fully observed before t |
| `clicks_90d` | Same — past 90 days of clicks |
| `avg_position_90d` | Average position over past 90 days, known before t |
| `ctr_90d` | Derived from past impressions/clicks (both known) |
| `content_age_days` | Static metadata from `dim_content` (content_created_date vs t) |
| `days_since_last_update` | Static metadata from `dim_content` (content_updated_date vs t) |
| `content_type` | Static metadata from `dim_content` |
| `has_ai_traffic` | Boolean: any `sessions_ai > 0` in past 90 days |


Correlations / Precision@50

In [14]:
import numpy as np
import pandas as pd

# One-hot encode content_type for correlations & modeling
df_encoded = pd.get_dummies(feature_frame, columns=['content_type'], prefix='content_type')

# 5 selected features + one-hot
cat_cols = [c for c in df_encoded.columns if c.startswith('content_type_')]
selected_feats = ['impressions_90d', 'clicks_90d', 'avg_position_90d', 'days_since_last_update'] + cat_cols

# Correlation with target
corr = df_encoded[selected_feats + ['decline_in_next_30d']].corr()['decline_in_next_30d'].sort_values()
print(corr)

# Univariate Precision@50
def precision_at_k(scores, labels, k=50):
    order = np.argsort(-scores)
    return labels[order[:k]].mean()

for feat in ['impressions_90d', 'clicks_90d', 'avg_position_90d', 'days_since_last_update']:
    p = precision_at_k(feature_frame[feat], feature_frame['decline_in_next_30d'])
    print(f"{feat}: Precision@50 = {p:.3f}")

# Categorical: use category mean rate as score
cat_means = feature_frame.groupby('content_type')['decline_in_next_30d'].mean()
print("Category mean decline rates:")
print(cat_means)

feature_frame['content_type_score'] = feature_frame['content_type'].map(cat_means)
p = precision_at_k(feature_frame['content_type_score'], feature_frame['decline_in_next_30d'])
print(f"content_type (via mean rate): Precision@50 = {p:.3f}")

content_type_keyword article      -0.080394
content_type_comparison article   -0.065856
avg_position_90d                  -0.043503
clicks_90d                        -0.001673
impressions_90d                    0.001938
days_since_last_update             0.063152
content_type_feedly article        0.132713
decline_in_next_30d                1.000000
Name: decline_in_next_30d, dtype: float64
impressions_90d: Precision@50 = 0.380
clicks_90d: Precision@50 = 0.280
avg_position_90d: Precision@50 = 0.100
days_since_last_update: Precision@50 = 0.380
Category mean decline rates:
content_type
comparison article    0.057489
feedly article        0.446320
keyword article       0.225244
Name: decline_in_next_30d, dtype: float64
content_type (via mean rate): Precision@50 = 0.560


Leakage Trap

Adding the label-window feature `imp_next30` makes the model appear perfect (Precision@50 = 1.000) because it gives the model the answer. The honest evaluation requires a client-holdout split on the clean feature set.


In [16]:
# Clean feature set (5 selected features, one-hot encoded)
selected_feats = ['impressions_90d', 'clicks_90d', 'avg_position_90d', 'days_since_last_update'] + \
                 [c for c in df_encoded.columns if c.startswith('content_type_')]

# Client-holdout evaluation
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier

groups = df_encoded['content_hash_id']
y = df_encoded['decline_in_next_30d']
gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
train_idx, test_idx = next(gss.split(df_encoded, y, groups))

def eval_model(cols, name):
    rf = RandomForestClassifier(n_estimators=100, min_samples_leaf=5,
                                random_state=42, n_jobs=-1)
    rf.fit(df_encoded.iloc[train_idx][cols], y.iloc[train_idx])
    s = rf.predict_proba(df_encoded.iloc[test_idx][cols])[:, 1]
    y_test = y.iloc[test_idx].to_numpy()
    print(f"{name}: P@50 = {precision_at_k(s, y_test):.3f} "
          f"(base rate = {y_test.mean():.3f})")

eval_model(selected_feats + ['imp_next30'], 'Leaky (with future)')
eval_model(selected_feats, 'Clean (client-holdout)')

Leaky (with future): P@50 = 1.000 (base rate = 0.233)
Clean (client-holdout): P@50 = 0.880 (base rate = 0.233)


Leaky P@50 = 1.000, Clean P@50 = 0.880 (client-holdout, base rate ≈ 0.23). The 12-point gap confirms the trap — future data inflates performance.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

1. **Intervention Bias**: Labels reflect post-editorial-action outcomes. Pages flagged and refreshed show "no decline" → model learns refreshed patterns don't decline. Mitigation: holdout set with no action.

2. **Warehouse Cutoff**: Max decision date with full label window = 2026-05-31. Production predictions (t ≥ 2026-06-01) have no warehouse labels for evaluation.

3. **Unbalanced Panel**: Client history depths differ (some 12mo, some 3mo). Client-holdout validates generalization to new clients, but per-client performance may vary.

4. **Sparse AI Signal**: `has_ai_traffic` has <0.1% positive rate — unlikely to contribute meaningfully.

5. **Future update dates in training data**: `days_since_last_update = -80` appears in the feature frame preview, indicating `content_updated_date > t` for some rows. These rows have information from the future relative to the decision date and should be filtered (`days_since_last_update >= 0`) in production.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.